# 01 · Data pipeline
**Owner:** _your name_ · Group 20 · Dutch Housing Blame

**Research question.** Who do people blame for the Dutch housing crisis online, and do Dutch-language
(r/thenetherlands) and English-language (r/Netherlands) communities blame different actors?

**Goal of this notebook.** Turn the raw weekly Arctic Shift files into clean, pseudonymised housing data that the rest of the group can use.

**Input:** `data/raw/{posts,comments}/<subreddit>/<week>.jsonl.zst` (from `collect.py`)
**Output:** `comments_housing.parquet`, `posts.parquet`, `author_activity.parquet` in `data/processed/`, plus descriptive figures.

**Done when:**
- [ ] all check cells pass
- [ ] processed files match the contract in the guide
- [ ] processed files uploaded to SURFdrive

**How to work:** each task has a few parts. A part says what has to exist when you're done, not how to build it –
split it into as many cells and functions as you like. Names in `code font` are used by the check cells or by other
notebooks, so keep them. *Go further* items are optional, for when we need more depth.

Read `notebooks/GUIDE.md` first (rules + the column names everyone uses).

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 20
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
ANNOTATION = ROOT / "annotation"
FIGURES = ROOT / "figures"
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

## Data collection
_Explain in your own words how `collect.py` works (weekly windows, paging, retries, why we did not use the keyword search) and add the completeness table: our counts vs. Arctic Shift's own counts._

## Task 1 · Load the raw data
**Why (for the paper):** The data section reports how much we collected and that it is complete.

**1A. Reading**
- A reusable way to read one week file (zstd-compressed JSON lines).
- All posts in `posts_raw` and all comments in `comments_raw`, with only the columns we need.
- 3 million comments is a lot of memory – think about how you build the DataFrame.

**1B. Completeness**
- `raw_counts`: a dict with the number of rows per (kind, subreddit), e.g. `{('posts', 'thenetherlands'): 38913, ...}`.
- 148 files per folder – check it.

In [ ]:
# check (do not change)
EXPECTED = {("posts", "thenetherlands"): 38913, ("posts", "Netherlands"): 79747,
            ("comments", "thenetherlands"): 1419203, ("comments", "Netherlands"): 1701447}
assert raw_counts == EXPECTED, raw_counts
print("ok")

**Go further (optional):**
- Comments per week per subreddit – any unusual weeks?
- Can you halve the memory use of `comments_raw`?

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 2 · Clean
**Why (for the paper):** Deleted content differs a lot between the subreddits; this goes into the data section and the limitations.

**2A. Drop what we can't use**
- Deleted, removed, empty and bot comments are dropped.
- `dropped`: a table with the number dropped per reason and subreddit.

**2B. Clean the text**
- Comment `text` and post titles without URLs, markdown link syntax, HTML entities or extra whitespace – test your cleaning on a few tricky examples first.
- `n_words` per comment.

In [ ]:
# check (do not change)
assert "subreddit" in dropped.columns or "subreddit" in dropped.index.names
print(dropped)

**Go further (optional):**
- Deleted/removed share per month per subreddit (plot).
- Usernames mentioned inside the text (`u/...`) replaced as well.

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 3 · Pseudonymise
**Why (for the paper):** Privacy & ethics is 4 points in the paper rubric – we must describe exactly how we protect users.

**3A. Hashing**
- Every username becomes `author_hash`: the first 16 characters of a salted SHA-256 hash.
- The salt lives in a `.env` file (`PSEUDONYM_SALT`), never in the notebook or in git, and is shared privately with the group.

**3B. Parents and clean-up**
- `parent_author_hash`: who wrote the comment or post that this comment replies to (needs *all* comments and posts, not only housing ones).
- No raw username left anywhere. The result is called `comments`.

In [ ]:
# check (do not change)
assert "author" not in comments.columns
assert comments["author_hash"].str.len().eq(16).all()
print("ok")

**Go further (optional):**
- Two sentences for the paper: why a salt, and why it stays out of git.

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 4 · Housing filter
**Why (for the paper):** A filter that catches non-housing comments weakens every result; the precision goes into the method section.

**4A. Keywords**
- A bilingual keyword list (Dutch + English). Start from the 12-week test list (woningnood, woningmarkt, huurwoning, sociale huur, verhuurder, huisjesmelker, housing, rental, landlord, tenant, ...) and decide as a group about `hypotheek`/`mortgage`.
- Dutch compounds (huurwoningen) must match too.

**4B. Flags**
- `kw_match` on comments, `housing_title` on posts, `in_housing_thread` on comments whose post has a housing title.
- `comments_housing`: comments with `kw_match` or `in_housing_thread`, with counts per subreddit.

**4C. Precision**
- `precision_sample`: 100 random `kw_match` comments that you labelled by hand (`is_housing` 1/0).
- Precision overall and per subreddit, a look at the false positives, and what you changed because of them.

In [ ]:
# check (do not change)
assert len(precision_sample) == 100 and precision_sample["is_housing"].notna().all()
print("precision:", precision_sample["is_housing"].mean())

**Go further (optional):**
- Which terms cause most false positives?
- Precision of comments that are only `in_housing_thread`.

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 5 · Context and language
**Why (for the paper):** Annotators need context, and since the communities share many users we also compare by language.

**5A. Parent context**
- `parent_text`: the parent comment's text, or the post title for top-level comments (empty if the parent is missing).
- How many parents are missing, and why?

**5B. Language**
- `lang` with `lingua`: `nl`, `en`, `mixed` or `other`. Your own rule for `mixed` and for very short texts – explain it.
- A manual check: does it look right on a random sample per language?

In [ ]:
# check (do not change)
assert set(comments_housing["lang"].unique()) <= {"nl", "en", "mixed", "other"}
print(comments_housing.groupby("subreddit")["lang"].value_counts(normalize=True).round(3))

**Go further (optional):**
- Share of English in r/thenetherlands and Dutch in r/Netherlands per quarter.

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 6 · User activity and descriptives
**Why (for the paper):** Notebook 04 needs the activity table; the data section needs a figure that shows the data at a glance.

**6A. User activity**
- `author_activity`: per user and subreddit the number of comments (all comments), housing comments, first and last date.
- Sanity check: roughly 76k users in r/thenetherlands and 168k in r/Netherlands.

**6B. Figures**
- Housing comments per month per subreddit, as counts and as a share of all comments.
- Language mix per subreddit.
- Saved as `figures/01_<name>.png` (`dpi=300`).

**Go further (optional):**
- Top 20 housing terms per subreddit (word counts, like in the A1 notebook).

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Save outputs
- `comments_housing`, `posts` and `author_activity` saved as parquet in `data/processed/`, with the columns from the guide.
- Uploaded to the SURFdrive processed folder, and the group knows.

In [ ]:
# check (do not change)
REQUIRED = ["id", "link_id", "parent_id", "subreddit", "created", "author_hash", "parent_author_hash",
            "score", "text", "parent_text", "kw_match", "in_housing_thread", "lang", "n_words"]
df = pd.read_parquet(DATA_PROC / "comments_housing.parquet")
missing = set(REQUIRED) - set(df.columns)
assert not missing, missing
assert not {"author", "parent_author"} & set(df.columns)
for f in ["posts.parquet", "author_activity.parquet"]:
    assert (DATA_PROC / f).exists(), f
print(len(df), "housing comments, ok")

## Notes for the paper
- Table: comments left after each step.
- Precision of the housing filter and the hypotheek/mortgage decision.
- How usernames are pseudonymised.